# DQMBot — Batch Image Query Driver

**Image layout expected:**
```
images/
    <subsystem>_<plotNumber>_<titleSlug>/
        <stem>[_grpN]_run<XXXXXX>.png
```

**Reference image layout:**
```
ref_images/
    <subsystem>/
        <subsystem>_<plotNumber>_<titleSlug>/
            <stem>[_grpN]_run<XXXXXX>.png
```

**Output layout (with run_id — preserves previous runs):**
```
results/
    <run_id>/
        <stem>/
            <stem>_<model>_run<XXXXXX>.txt
        summary_<run_id>.csv
```

**Output layout (no run_id — overwrites):**
```
results/
    <stem>/
        <stem>_<model>_run<XXXXXX>.txt
    summary.csv
```

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from owui_client import (
    ModelConfig, RunMetadata, BatchConfig,
    DirectImages, CaptionedReferences, CaptionedBoth,
    list_models, validate_models,
    build_messages, send_query, query, resolve_prompt,
    caption_image, resolve_image_inputs,
    run_batch, retry_failed, sanity_check,
    batch_query_images, _collect_images, resolve_output_dir, resolve_output_file,
    find_reference_image, find_reference_images,
)
from rag_backends import (
    LocalRAG, YAMLContext, OWUIContext, NoContext,
    retrieve_and_inspect,
)

print('owui_client loaded OK')

In [ ]:
# ── Dependencies (run once, then restart kernel) ───────────────────────────────
# !pip install rank-bm25 langchain-huggingface sentence-transformers --quiet

In [ ]:
# ── Discover available models ──────────────────────────────────────────────────
print('=== Models ===')
for m in list_models():
    print(' ', m)

In [ ]:
# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are an assistant to CMS detector operations shifters, helping evaluate DQM monitoring plots.

Assess whether the input plot indicates a detector problem requiring action.

Respond in 4 sections:

Instructions: Quote the shift instructions most relevant to this plot type.

Observations: Describe what you see in the input plot (color distribution, notable features, anomalies). If a reference plot is shown, describe it and note how the input differs.

Assessment: Compare the input plot against the quoted instructions. If a reference is provided, use it to calibrate what normal looks like.

Verdict: State GOOD or BAD. If BAD, specify the required action from the instructions (e.g. "contact ECAL DOC", "add to elog only").\
"""

In [ ]:
# ── Batch configuration ───────────────────────────────────────────────────────
cfg = BatchConfig(
    run_id      = 'YAML',
    image_root  = Path('images'),
    output_root = Path('results'),
    ref_dir     = Path('ref_images'),
    plot_filter = None,             # None — run all plots
    # Models: plain string, or ModelConfig(name=..., image_token_budget=...) for Gemma 4
    # Valid image_token_budget values: 70, 140, 280, 560, 1120
    models = [
        # 'qwen2.5vl:latest',             #7b
        # 'qwen2.5vl:32b',                #32b
        # 'qwen3-vl:latest',              #8b
        # 'gemma3:latest',                #4b
        # "qwen/qwen35-9b",
        "qwen/qwen3.6",                                                   #35b
        'google/gemma3-27b',                                              #27b
        'google/gemma4-31b',                                              #31b, default budget
        # ModelConfig(name='google/gemma4-31b', image_token_budget=140), #31b, low budget
    ],
    # Pick one context backend:
    # context = LocalRAG(csv_path=Path('document_chunks.csv')),  # BM25 + vector search over CSV
    context = YAMLContext(),                                      # direct lookup from plot_instructions/
    # context = OWUIContext(collection_ids=['<id>']),             # server-side retrieval via OWUI
    # context = NoContext(),                                      # no retrieval
    # Pick one image mode — controls whether images are sent to the model directly or
    # captioned first. Captioning defaults to self-caption (each model under test captions
    # its own images); pass caption_model='...' to use one fixed captioner instead. Captions
    # used are recorded in each result .txt file — there is no separate caption cache.
    image_mode = DirectImages(),                                  # send both reference + input images directly (default)
    # image_mode = CaptionedReferences(),                         # caption references, input image sent directly
    # image_mode = CaptionedBoth(),                                # caption both references and input image
    # image_mode = CaptionedBoth(caption_model='google/gemma3-27b'),  # use one fixed captioner instead of self-caption
    system_prompt = SYSTEM_PROMPT,
    prompt        = "",
    delay         = 1.5,
    run_metadata  = RunMetadata(event_type_map={
        398185: 'collisions',
        398186: 'cosmics',
        398187: 'circulating',
        398188: 'collisions',
        398189: 'collisions',
        398191: 'collisions',
        398194: 'cosmics',
        398199: 'cosmics',
    }),
)

pairs = cfg.build_pairs()
print(f'{len(pairs)} images across {len(set(p for p, _ in pairs))} plots')
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── Example: load a BatchConfig from a previous run's JSON instead ────────────
# Every run_batch() call writes results/<run_id>/config_<run_id>.json — reload it
# to reproduce or extend that exact run without redefining cfg by hand.
#
# cfg = BatchConfig.from_json('results/YAML/config_YAML.json')
# pairs = cfg.build_pairs()
# print(f'{len(pairs)} images across {len(set(p for p, _ in pairs))} plots')

In [ ]:
# ── Debug: inspect what RAG retrieves for the query ──────────────────────────
# Only applies when cfg.context is a LocalRAG instance.
debug_query = "ECal TP ET-weighted Occupancy"
hits = retrieve_and_inspect(debug_query, csv_path=cfg.context.csv_path, top_k=5, method="hybrid")

print(f'Query: "{debug_query}"\n')
for h in hits:
    print(f"  #{h['rank']}  score={h['score']:.4f}  doc={h['document_id']}  "
          f"chunk={h['chunk_index']}  len={h['text_length']}")
    print(f"       source: {h['source']}")
    print(f"       preview: {h['text_preview'][:120]}...")
    print()

In [ ]:
# ── Sanity check ──────────────────────────────────────────────────────────────
sanity_check(cfg, pairs)

In [ ]:
# ── Smoke test: one image, first model ───────────────────────────────────────
if pairs:
    plot_name, img = pairs[0]
    refs = find_reference_images(img, cfg.ref_dir) if cfg.ref_dir and Path(cfg.ref_dir).exists() else []

    _m = cfg.models[0]
    smoke_model = _m if isinstance(_m, ModelConfig) else ModelConfig(name=_m)

    resolved_prompt = resolve_prompt(cfg.prompt, img, cfg.run_metadata)
    spec = build_messages(
        resolved_prompt,
        system=cfg.system_prompt,
        reference_images=refs,
        image_path=img,
        context=cfg.context,
    )

    print(f"Plot            : {plot_name}")
    print(f"Image           : {img.name}")
    print(f"References      : {[r.name for r in spec['ref_list']]}")
    print(f"RAG backend     : {type(spec['context']).__name__}")
    print(f"Prompt          : {repr(spec['prompt'])}")
    print(f"RAG text ({len(spec['rag_text'])} chars):")
    print(spec['rag_text'][:600] or '  (none)')
    print()

    test = send_query(spec, model=smoke_model)
    print(f"Model      : {test['model']}")
    for f in smoke_model.__dataclass_fields__:
        if f != 'name' and getattr(smoke_model, f) is not None:
            print(f"  {f}: {getattr(smoke_model, f)}")
    print(f"Load       : {test['load_latency_s']}s")
    print(f"Generation : {test['generation_latency_s']}s")
    print(f"Total      : {test['latency_s']}s")
    print(f"Error      : {test['error']}")
    print()
    print(test['response'])

### Run on all images

In [ ]:
# ── Full batch ────────────────────────────────────────────────────────────────
results = run_batch(cfg, pairs,overwrite=False)
print(f'\nDone. {len(results)} queries completed.')

In [ ]:
# ── Retry errors ──────────────────────────────────────────────────────────────
results = retry_failed(cfg, results)

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
import re as _re

df = pd.DataFrame(results)
df['image_name'] = df['image'].apply(lambda p: Path(p).name if p else None)

errors = df[df['error'].notna()]
if not errors.empty:
    print(f'WARNING: {len(errors)} failed queries:')
    display(errors[['plot_name', 'model', 'image_name', 'error']])
else:
    print('All queries succeeded.')

print()
display(
    df.groupby(['plot_name', 'model'])[['load_latency_s', 'generation_latency_s', 'latency_s']]
      .mean()
      .round(2)
      .rename(columns={'load_latency_s': 'load_s', 'generation_latency_s': 'gen_s', 'latency_s': 'total_s'})
)

In [ ]:
# ── Save CSV next to the run's output folder (append, don't overwrite) ────────
if cfg.run_id:
    csv_path = cfg.output_root / cfg.run_id / f'summary_{cfg.run_id}.csv'
else:
    csv_path = cfg.output_root / 'summary.csv'

if csv_path.exists():
    df_to_save = pd.concat([pd.read_csv(csv_path), df], ignore_index=True)
else:
    df_to_save = df

df_to_save.to_csv(csv_path, index=False)
print(f'Saved: {csv_path} (+{len(df)} rows, {len(df_to_save)} total)')

# Show result tree
print()
root = cfg.output_root / cfg.run_id if cfg.run_id else cfg.output_root
for item in sorted(root.iterdir()):
    if item.is_dir():
        txts = list(item.glob('*.txt'))
        print(f'  {item.name}/  ({len(txts)} files)')
        for t in sorted(txts):
            print(f'    {t.name}')
    elif item.suffix == '.csv':
        print(f'  {item.name}')

In [ ]:
# ── Side-by-side comparison: one image across all models ─────────────────────
COMPARE_PLOT  = pairs[0][0]
COMPARE_IMAGE = pairs[0][1].name

subset = df[(df['plot_name'] == COMPARE_PLOT) & (df['image_name'] == COMPARE_IMAGE)]
for _, row in subset.iterrows():
    print('=' * 72)
    print(f"Model   : {row['model']}")
    print(f"Latency : {row['latency_s']}s")
    print()
    if row['error']:
        print(f"ERROR: {row['error']}")
    else:
        print(row['response'])
    print()